# Questão 2 — ID3, C4.5 e CART (implementação própria)

Este notebook atende ao item 2 da lista: construir as **árvores de decisão** e apresentar
suas **bases de regras**, usando os algoritmos **ID3**, **C4.5** e **CART**.

Em vez de reimplementar tudo do zero, este notebook **reaproveita as classes que já foram
implementadas manualmente na Questão 1** (pasta `Questao_1/Algoritimos`). Aqui elas são
apenas importadas, aplicadas na mesma base de dados e adaptadas para gerar 3 formatos de
saída diferentes, explicados na seção 3.

## 1. Tratamento da base de dados

O tratamento é o mesmo feito na Questão 1: a base (`database.csv`) tem uma coluna
`Exemplo` (E1, E2, E3, ...) que é apenas um **índice/identificador da linha**, sem
nenhum valor preditivo. Ela é removida antes de treinar as árvores.

In [1]:
import sys
import os
import pandas as pd

# Pasta onde estão as implementações da Questão 1 (reaproveitadas aqui)
ALGORITHMS_FOLDER = os.path.join("..", "Questao_1", "Algoritimos")
sys.path.append(ALGORITHMS_FOLDER)

# Base de dados original, a mesma usada na Questão 1
original_data = pd.read_csv(os.path.join(ALGORITHMS_FOLDER, "database.csv"))

# Tratamento: remover a coluna 'Exemplo'
original_data = original_data.drop(columns=["Exemplo"])

os.makedirs("outputs", exist_ok=True)
original_data.to_csv(os.path.join("outputs", "database_tratada.csv"), index=False)

original_data.head()

,Historia de Credito,Divida,Garantia,Renda,Tempo de Emprego,Risco
0,Ruim,Alta,Nenhuma,$0 a $15k,Menos de 1 ano,Alto
1,Desconhecida,Alta,Nenhuma,$15 a $35k,Menos de 1 ano,Alto
2,Desconhecida,Baixa,Nenhuma,$15 a $35k,1 a 3 anos,Moderado
3,Desconhecida,Baixa,Nenhuma,$0 a $15k,Menos de 1 ano,Alto
4,Desconhecida,Baixa,Nenhuma,Acima de $35k,Acima de 3 anos,Baixo


## 2. Importando as árvores implementadas na Questão 1

As classes `id3`, `c45` e `cart` (e a classe auxiliar `Node`) estão em
`Questao_1/Algoritimos/*.py`. Adicionamos essa pasta ao `sys.path` e importamos
normalmente, sem duplicar código.

In [2]:
from id3 import id3
from c45 import c45
from cart import cart
from node import Node

print("Classes importadas de:", os.path.abspath(ALGORITHMS_FOLDER))

Classes importadas de: C:\Users\PedroGomes\Documents\5 Periodo\IA\Lista AB1\Questao_1\Algoritimos


## 3. Três formatos de saída (adaptação feita nesta questão)

As classes foram adaptadas para aceitar/gerar **3 formatos de saída** a partir da mesma
árvore treinada, cobrindo os três usos que precisamos: apresentar o processo de
construção, apresentar a árvore final, e avaliar o modelo.

1. **Log completo ** (`export_tree`): foi realizado para auxiliar na correção 
do cálculos das árvores manulamente — só é gerado se `verbose=True`.
2. **Árvore em texto pura** (`export_pure_tree`): só a estrutura da árvore, sem o log —
   é esse texto que serve de saída "oficial" da árvore para o relatório.
3. **CSV de classificação** (`classify_dataset`): aplica a árvore em cada linha da base
   e gera um csv com as colunas `Real` x `Predito`, pronto para calcular métricas como
   F-score.

Abaixo estão os trechos reais de `id3.py` (idênticos em `c45.py`/`cart.py`), copiados e
comentados, mostrando como cada uma dessas 3 saídas é gerada:

```python
# --- id3.py ---

def export_tree(self, filepath: str) -> None:
    """Escreve toda a saída (log detalhado + árvore final) em um único arquivo de texto.

    Só gera o arquivo se verbose=True; sem verbose não há nada a exportar."""
    if not self.verbose:
        return  # <- sem verbose, nada é impresso/gerado (só classificação continua ativa)

    content = []
    content.append("===== LOG DE CONSTRUÇÃO (ID3) =====\n")
    content.extend(self.log)              # log acumulado durante o build_tree (entropia, ganhos, nó escolhido...)
    content.append("\n===== ÁRVORE FINAL (ID3) =====\n")
    content.extend(self._tree_text())      # desenha a árvore final percorrendo os Nodes

    with open(filepath, "w", encoding="utf-8") as f:
        f.write("\n".join(content))


def export_pure_tree(self, filepath: str = None) -> str:
    """Gera apenas a árvore em texto (sem o log de construção)."""
    text = "\n".join(self._tree_text())   # reaproveita o mesmo desenho de árvore

    if filepath:
        with open(filepath, "w", encoding="utf-8") as f:
            f.write(text)

    return text  # <- é este texto puro que vira a "saída oficial" da árvore


def classify_dataset(self, df: pd.DataFrame = None, filepath: str = None) -> pd.DataFrame:
    """Classifica a base e retorna (real x predito), pronto para avaliação (ex: f-score)."""
    if df is None:
        df = self.df

    result = pd.DataFrame({
        "Real": df[self.target],
        "Predito": df.apply(self.predict, axis=1),  # percorre a árvore linha a linha
    })

    if filepath:
        result.to_csv(filepath, index=False)

    return result
```

`_tree_text` é o método comum que desenha a árvore (usado pelas duas primeiras
saídas), percorrendo recursivamente os `Node`:

```python
def _tree_text(self, node: Node = None, depth: int = 0, lines: list = None) -> list:
    is_root = lines is None
    if is_root:
        lines = []
    if node is None:
        node = self.root

    prefix = "  " * depth
    if node.is_leaf():
        lines.append(f"{prefix}[Folha] Classe: {node.label}")
    else:
        lines.append(f"{prefix}[Nó] Atributo: {node.attribute}")
        for value, child in node.children.items():
            lines.append(f"{prefix}  -> Valor \'{value}\':")
            self._tree_text(child, depth + 2, lines)  # desce para os filhos

    return lines
```

## 4. Onde está a parte principal de cada algoritmo

As três classes têm a mesma estrutura (`build_tree`, `best_attribute`, `predict`...); a
única diferença real entre ID3, C4.5 e CART é a **métrica usada para escolher o melhor
atributo** em cada nó. Os trechos abaixo, copiados de cada arquivo, mostram exatamente
essa diferença.

**ID3 — Ganho de Informação** (`id3.py`): escolhe o atributo que mais reduz a
entropia do conjunto.

```python
def calc_entropy_gain(self, attribute: str, df: pd.DataFrame = None) -> float:
    general = self.calc_general_entropy(df)            # entropia do conjunto atual
    attr_entropy = self.calc_attribute_entropy(attribute, df)  # entropia média após dividir pelo atributo
    gain = general - attr_entropy                        # ganho = quanto a entropia caiu
    return gain
```

**C4.5 — Ganho de Razão** (`c45.py`): igual ao ID3, mas normaliza o ganho pela
"quantidade de divisão" (`split info`) do atributo, evitando favorecer atributos com
muitos valores distintos.

```python
def calc_gain_ratio(self, attribute: str, df: pd.DataFrame = None) -> float:
    general = self.calc_general_entropy(df)
    attr_entropy = self.calc_attribute_entropy(attribute, df)
    gain = general - attr_entropy                         # mesmo ganho de informação do ID3

    split_info = self.calc_split_info(attribute, df)       # entropia da distribuição dos valores do atributo
    gain_ratio = gain / split_info if split_info != 0 else 0
    return gain_ratio                                      # ganho "normalizado"
```

**CART — Índice Gini** (`cart.py`): em vez de entropia, usa o índice Gini como medida
de impureza; escolhe o atributo que mais reduz o Gini médio ponderado.

```python
def calc_gini_gain(self, attribute: str, df: pd.DataFrame = None) -> float:
    general = self.calc_gini(df)                          # gini do conjunto atual
    attr_gini = self.calc_attribute_gini(attribute, df)     # gini médio ponderado após dividir
    gain = general - attr_gini
    return gain
```

Nos três casos, `best_attribute` calcula o ganho para todos os atributos restantes e
escolhe o de maior valor — só troca a fórmula de ganho usada:

```python
def best_attribute(self, attributes: list, df: pd.DataFrame = None) -> tuple:
    gains = {attr: self.calc_entropy_gain(attr, df) for attr in attributes}  # (troca por calc_gain_ratio/calc_gini_gain)
    chosen = max(gains, key=gains.get)
    return chosen, gains
```

## 5. Treinando e gerando as saídas — ID3

In [3]:
id3_tree = id3(original_data, verbose=True)
id3_tree.fit("Risco")


# árvore em texto pura (essa é a "saída oficial" da árvore)
id3_tree_text = id3_tree.export_pure_tree(os.path.join("outputs", "id3_arvore.txt"))
print(id3_tree_text)

# csv de classificação (real x predito), para avaliação (ex: f-score)
id3_predictions = id3_tree.classify_dataset(filepath=os.path.join("outputs", "id3_predicoes.csv"))
id3_predictions.head()

[Nó] Atributo: Tempo de Emprego
  -> Valor 'Menos de 1 ano':
    [Nó] Atributo: Renda
      -> Valor '$0 a $15k':
        [Nó] Atributo: Garantia
          -> Valor 'Nenhuma':
            [Nó] Atributo: Historia de Credito
              -> Valor 'Ruim':
                [Folha] Classe: Alto
              -> Valor 'Desconhecida':
                [Folha] Classe: Alto
              -> Valor 'Boa':
                [Nó] Atributo: Divida
                  -> Valor 'Alta':
                    [Folha] Classe: Alto
                  -> Valor 'Baixa':
                    [Folha] Classe: Moderado
          -> Valor 'Adequada':
            [Folha] Classe: Moderado
      -> Valor '$15 a $35k':
        [Folha] Classe: Alto
      -> Valor 'Acima de $35k':
        [Folha] Classe: Moderado
  -> Valor '1 a 3 anos':
    [Nó] Atributo: Historia de Credito
      -> Valor 'Desconhecida':
        [Folha] Classe: Moderado
      -> Valor 'Boa':
        [Nó] Atributo: Divida
          -> Valor 'Alta':
          

,Real,Predito
0,Alto,Alto
1,Alto,Alto
2,Moderado,Moderado
3,Alto,Alto
4,Baixo,Baixo


## 6. Treinando e gerando as  saídas — C4.5

In [4]:
c45_tree = c45(original_data, verbose=True)
c45_tree.fit("Risco")

c45_tree_text = c45_tree.export_pure_tree(os.path.join("outputs", "c45_arvore.txt"))
print(c45_tree_text)

c45_predictions = c45_tree.classify_dataset(filepath=os.path.join("outputs", "c45_predicoes.csv"))
c45_predictions.head()

[Nó] Atributo: Tempo de Emprego
  -> Valor 'Menos de 1 ano':
    [Nó] Atributo: Garantia
      -> Valor 'Nenhuma':
        [Nó] Atributo: Divida
          -> Valor 'Alta':
            [Folha] Classe: Alto
          -> Valor 'Baixa':
            [Nó] Atributo: Renda
              -> Valor '$0 a $15k':
                [Nó] Atributo: Historia de Credito
                  -> Valor 'Desconhecida':
                    [Folha] Classe: Alto
                  -> Valor 'Ruim':
                    [Folha] Classe: Alto
                  -> Valor 'Boa':
                    [Folha] Classe: Moderado
              -> Valor 'Acima de $35k':
                [Folha] Classe: Moderado
      -> Valor 'Adequada':
        [Folha] Classe: Moderado
  -> Valor '1 a 3 anos':
    [Nó] Atributo: Historia de Credito
      -> Valor 'Desconhecida':
        [Folha] Classe: Moderado
      -> Valor 'Boa':
        [Nó] Atributo: Divida
          -> Valor 'Alta':
            [Folha] Classe: Moderado
          -> Valor 'Bai

,Real,Predito
0,Alto,Alto
1,Alto,Alto
2,Moderado,Moderado
3,Alto,Alto
4,Baixo,Baixo


## 7. Treinando e gerando as  saídas — CART

In [5]:
cart_tree = cart(original_data, verbose=True)
cart_tree.fit("Risco")

cart_tree_text = cart_tree.export_pure_tree(os.path.join("outputs", "cart_arvore.txt"))
print(cart_tree_text)

cart_predictions = cart_tree.classify_dataset(filepath=os.path.join("outputs", "cart_predicoes.csv"))
cart_predictions.head()

[Nó] Atributo: Tempo de Emprego
  -> Valor 'Menos de 1 ano':
    [Nó] Atributo: Renda
      -> Valor '$0 a $15k':
        [Nó] Atributo: Garantia
          -> Valor 'Nenhuma':
            [Nó] Atributo: Historia de Credito
              -> Valor 'Ruim':
                [Folha] Classe: Alto
              -> Valor 'Desconhecida':
                [Folha] Classe: Alto
              -> Valor 'Boa':
                [Nó] Atributo: Divida
                  -> Valor 'Alta':
                    [Folha] Classe: Alto
                  -> Valor 'Baixa':
                    [Folha] Classe: Moderado
          -> Valor 'Adequada':
            [Folha] Classe: Moderado
      -> Valor '$15 a $35k':
        [Folha] Classe: Alto
      -> Valor 'Acima de $35k':
        [Folha] Classe: Moderado
  -> Valor '1 a 3 anos':
    [Nó] Atributo: Historia de Credito
      -> Valor 'Desconhecida':
        [Folha] Classe: Moderado
      -> Valor 'Boa':
        [Nó] Atributo: Divida
          -> Valor 'Alta':
          

,Real,Predito
0,Alto,Alto
1,Alto,Alto
2,Moderado,Moderado
3,Alto,Alto
4,Baixo,Baixo
